# Exercise 2: Vowel harmony and consonant alternation

*The Computational Phenotype · version 0.1.0*

> Add the plural suffix to *ev*, *araba*, *göz* and *kol*. Separately, form the third-person singular past of *gel-*, *bak-*, *gör-*, *bul-* and *git-*. Identify which changes follow vowel harmony and which come from consonant assimilation. Compare a regular example with one of the exceptional stems in the vowel-harmony section.

The notebook implements the two rules in a few lines and checks them against the Apertium Turkish generator.

**Setup.** `pip install "turkicnlp[hfst]==0.5.0" pandas`. Download the language resources once with `turkicnlp.download(lang, processors=[...])`, as shown in the first code cell. Cells that fetch data from the web need internet access.

In [ ]:
import hfst
import turkicnlp
from turkicnlp.resources.registry import ModelRegistry

turkicnlp.download("tur", processors=["morph"])          # Apertium Turkish transducers (once)
tur_dir = ModelRegistry.get_model_path("tur", "morph", "apertium", script="Latn")

def load(path):
    stream = hfst.HfstInputStream(str(path)); fst = stream.read(); stream.close()
    fst.convert(hfst.ImplementationType.HFST_OLW_TYPE)   # optimized lookup: no epsilon symbols in output
    return fst

analyser = load(tur_dir / "tur.automorf.hfst")
generator = load(tur_dir / "tur.autogen.hfst")
analyse = lambda word: sorted({a for a, _ in analyser.lookup(word)})
generate = lambda analysis: sorted({f for f, _ in generator.lookup(analysis)})

In [ ]:
FRONT, BACK = set("eiöü"), set("aıou")
ROUNDED = set("öüou")
VOICELESS = set("çfhkpsşt")

def last_vowel(stem):
    return next(c for c in reversed(stem) if c in FRONT | BACK)

def plural(stem):                                    # 2-way harmony: -lAr
    return stem + ("ler" if last_vowel(stem) in FRONT else "lar")

def past_3sg(stem):                                  # 4-way harmony and voicing: -DI
    v = last_vowel(stem)
    vowel = {(True, False): "i", (True, True): "ü", (False, False): "ı", (False, True): "u"}[(v in FRONT, v in ROUNDED)]
    consonant = "t" if stem[-1] in VOICELESS else "d"    # consonant assimilation
    return stem + consonant + vowel

import pandas as pd
rows = []
for stem in ["ev", "araba", "göz", "kol"]:
    rows.append({"form": "plural", "stem": stem, "rule": plural(stem), "Apertium": ", ".join(generate(f"{stem}<n><pl><nom>"))})
for stem, valency in [("gel", "iv"), ("bak", "iv"), ("gör", "tv"), ("bul", "tv"), ("git", "iv")]:
    rows.append({"form": "past 3sg", "stem": stem, "rule": past_3sg(stem), "Apertium": ", ".join(generate(f"{stem}<v><{valency}><ifi><p3><sg>"))})
pd.DataFrame(rows)

In [ ]:
# Exceptional stems: final consonant palatalised, so front suffixes follow a back vowel
for stem in ["saat", "rol", "hal"]:
    print(f"{stem:<5} rule: {plural(stem):<9} Apertium plural: {generate(stem + '<n><pl><nom>')}  accusative: {generate(stem + '<n><acc>')}")

## Worked solution

| Stem | Plural | Explanation |
|---|---|---|
| *ev* | *evler* | front vowel → *-ler* |
| *araba* | *arabalar* | back vowel → *-lar* |
| *göz* | *gözler* | front (rounded) vowel → *-ler*; the plural has only two forms |
| *kol* | *kollar* | back vowel → *-lar* |

| Stem | Past 3sg | Vowel harmony | Consonant assimilation |
|---|---|---|---|
| *gel-* | *geldi* | front unrounded → *i* | voiced *l* → *d* |
| *bak-* | *baktı* | back unrounded → *ı* | voiceless *k* → *t* |
| *gör-* | *gördü* | front rounded → *ü* | voiced *r* → *d* |
| *bul-* | *buldu* | back rounded → *u* | voiced *l* → *d* |
| *git-* | *gitti* | front unrounded → *i* | voiceless *t* → *t* |

The vowel of the suffix follows harmony (two-way for *-lAr*, four-way for *-DI*); the initial consonant of *-DI* follows the voicing of the stem's last sound. The rules reproduce all nine Apertium forms.

**Exceptions.** *saat* 'hour' has a back vowel but a palatalised final *t*, so it takes front suffixes: the plural is *saatler*, not *saatlar*, which the simple rule above wrongly predicts. Apertium lists *saat* as exceptional and generates *saatler* and *saati*. The run also shows a gap in the transducer: for *rol* 'role' it generates the accusative *roli* and does not recognise *rolü*, the standard spelling (palatalised *l* plus rounding after *o*). Exceptional stems have to be listed in a lexicon, and the lexicon itself has to be checked.